## 依赖注入
核心思想：使用依赖注入来共享通用逻辑，避免代码重复

如果这段逻辑满足`「多个地方要用」`+`「与具体业务无关」`+`「失败就该中断请求」`三条中的两条以上，就抽成依赖。
- 部分接口需要分页参数
- 部分接口需要认证
- 数据库会话管理


```
请求 → Middleware A → Middleware B
        ↓
     路由匹配（找不到 = 404，直接原路返回）
        ↓
   全局 dependencies  [FastAPI(dependencies=...)] 【全部路由】
        ↓
   挂载级 / Router 级 dependencies 【该 router 下全部接口】
        ↓
   装饰器级 dependencies 【仅当前接口】
        ↓
   函数参数级 Depends（含子依赖，自动去重） 【仅当前接口】
        ↓
      业务 handler
        ↓
   yield 清理（内层→外层）
        ↓
     Middleware B → Middleware A  ← 响应后处理

其中中间件定义顺序：后注册的再外面，先注册的再里面，所以是先定义 Middleware B， 后定义 Middleware A
```

### 1. 依赖注入写法

#### 函数依赖

In [ ]:
from fastapi import FastAPI, Query,Depends
app = FastAPI()

@app.get("/")
def root():
    return {"message": "Hello World"}

# 分页参数逻辑共用：商品列表和用户列表
def common_params(
    skip: int = Query(0, ge=0),
    limit: int = Query(10, le=60)
) -> dict:
    return {"skip": skip, "limit": limit}

@app.get("/item/item_list")
def get_item_list(commons: dict = Depends(common_params)):
    return {"commons": commons}

@app.get("/user/user_list")
def get_user_list(commons: dict = Depends(common_params)):
    return {"commons": commons}

`Depends` 做了什么：当请求到达时，FastAPI 先调用 `common_params`，拿到返回值，然后把它注入到 `get_xxx_list` 的 `commons` 参数中

#### 类依赖

In [ ]:
class CommonQueryParams:
    def __init__(self, q: str = None, skip: int = 0, limit: int = 100):
        self.q = q
        self.skip = skip
        self.limit = limit

@app.get("/items/")
async def read_items(commons: CommonQueryParams = Depends()):
    return commons

#### 子依赖链: 依赖的依赖

依赖函数本身也可以依赖其他依赖，形成一条链。这是实现认证系统的推荐方式。

In [ ]:
# 第一层：从 Header 中提取并校验 Token
async def verify_token(authorization: str = Header(...)):
    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="无效的认证头")
    token = authorization.replace("Bearer ", "")
    return token


# 第二层：基于已验证的 Token 查询当前用户
async def get_current_user(token: str = Depends(verify_token)):
    # 这里应该是真实的数据库查询，暂用模拟数据
    user = fake_db.get_user_by_token(token)
    if not user:
        raise HTTPException(status_code=401, detail="用户不存在")
    return user


# 第三层：检查用户是否是管理员
async def require_admin(user: dict = Depends(get_current_user)):
    if user.get("role") != "admin":
        raise HTTPException(status_code=403, detail="权限不足")
    return user


# 路由中只需要声明最终依赖
@app.get("/admin/dashboard")
async def admin_dashboard(admin: dict = Depends(require_admin)):
    return {"message": f"欢迎，{admin['username']}管理员"}

执行流程是：`verify_token` → `get_current_user` → `require_admin` → 路由函数。每一层只关心自己的职责，像乐高积木一样组合。

#### 路径操作装饰器依赖

如果你只需要依赖的"副作用"（比如权限校验），不需要它的返回值，可以在路由的 ``dependencies` 参数中声明：

In [ ]:
from fastapi import HTTPException, Header
async def verify_token(x_token: str = Header(...)):
    if x_token != "fake-super-secret-token":
        raise HTTPException(status_code=400, detail="X-Token header invalid")

@app.get("/items/", dependencies=[Depends(verify_token)])
async def read_items():
    return [{"item": "Foo"}, {"item": "Bar"}]

### 2. yield 依赖与资源生命周期

有些资源需要"用完就关"——数据库连接、文件句柄等。FastAPI 的 yield 依赖完美解决了这个问题：

In [ ]:
def get_db():
    db = SessionLocal()
    try:
        yield db          # 把 db 交给路由函数使用
    finally:
        db.close()        # 路由函数执行完毕后，无论成功还是异常，都会执行这里


@app.get("/users/{user_id}")
async def read_user(user_id: int, db: Session = Depends(get_db)):
    user = db.query(User).filter(User.id == user_id).first()
    return user

`yield` 之前的代码是"预处理"（创建资源），`yield` 之后的代码是"后处理"（清理资源）。这本质上是一个上下文管理器模式。

> 注意：不要在 `yield`之后写业务逻辑，因为那些代码会在响应返回后才执行。

### 3. 依赖注入应用场景
- 处理请求参数：从请求中提取参数和验证参数（路径参数、查询参数、请求体）
- 共享业务逻辑：抽取封装多个路由公用的逻辑代码
- 共享数据库连接：管理数据库会话的创建、使用、关闭
- 安全和认证：验证用户身份、检查权限和角色要求等

| 依赖模式 | 典型写法 | 适用场景 |
|---|---|---|
| 函数依赖 | `def common_params(...): return {...}` | 抽取分页、查询参数、公共校验 |
| 数据库会话 | `def get_db(): ... yield db ...` | 每个请求获取并释放 DB 会话 |
| 用户认证 | `def get_current_user(token=Header(...))` | 解析 Token、获取当前用户 |
| 权限校验 | `def require_admin(user=Depends(get_current_user))` | 角色/权限检查，失败直接拦截 |
| 子依赖/依赖链 | 依赖函数里继续用 `Depends()` | 先取会话，再取用户，再校验权限 |
| 类依赖 | `class ItemService: ...` | 封装服务层、Repository、业务逻辑 |
| 参数化依赖 | `__call__` 实例或闭包工厂 | 传入固定配置、阈值、资源名等 |
| 全局/路由组依赖 | `FastAPI(dependencies=[...])` | 全局鉴权、日志、限流等副作用 |
| 测试覆盖依赖 | `app.dependency_overrides[...]` | 用假 DB、假用户替换真实依赖 |




